# Mathematical Engineering - Financial Engineering, FY 2025-2026

# Risk Management - Assignment 1: Hedging a Swaption Portfolio

**Case study:** The IR-derivative desk of Polimi Bank has the following positions opened today (15/02/2008):

- Long swaption payer 1m&10y 5y ATM - Notional €650 Mln
- Vanilla 10y IRS fixed rate receiver - Notional €550 Mln


In [27]:
# Importing the libraries
import numpy as np
import pandas as pd
import pickle
import datetime as dt

from date_functions import (
    business_date_offset,
    date_series,
    year_frac_act_x
)
from ex0_utilities import bootstrap
from ex1_utilities import (
    swaption_price_calculator,
    irs_proxy_duration,
    swap_par_rate,
    swap_mtm,
    SwapType,
)

In [28]:
# =============================================================================
# Portfolio Parameters
# =============================================================================

# Swaption
swaption_maturity_y = 10  # Years component of swaption expiry
swaption_maturity_m = 1  # Months component of swaption expiry
swaption_tenor_y = 5  # Underlying swap tenor in years
swaption_fixed_leg_freq = 1  # Annual fixed leg payments
swaption_type = SwapType.PAYER
swaption_notional = 650_000_000
sigma_black = 0.7955  # Black implied volatility for the swaption

# IRS
irs_maturity = 10  # In years
irs_fixed_leg_freq = 1  # Annual fixed leg payments
irs_notional = 550_000_000
irs_type = SwapType.RECEIVER #ERRORE ERA PAYER

In [29]:
dt = pd.read_csv('dt.csv',
                index_col = 'Market',
                usecols = ['Market','TARGET'],
                converters = {'TARGET':pd.to_datetime})

depo_converter = lambda x: float(x)
df_depos = pd.read_csv('depos.csv', 
                   index_col ='Depos',
                   usecols = ['Depos','ASK','BID'], 
                    converters={'Depos':pd.to_datetime,'BID':depo_converter,'ASK':depo_converter})

future_converter = lambda x: float(x)
futures = pd.read_csv('futures.csv',
                      index_col ='Futures',
                      usecols = ['Futures','ASK','BID'],
                      converters = {'Futures':pd.to_datetime, 'Settle':pd.to_datetime, 'Expiry':pd.to_datetime})
expiry = pd.read_csv('expiry.csv',
                     index_col = 'Futures',
                     usecols =['Futures', 'Settle', 'Expiry'], 
                     converters = {'Futures':pd.to_datetime, 'Settle':pd.to_datetime, 'Expiry':pd.to_datetime})
df_futures = futures.join(expiry)

swap_converter = lambda x: float(x)
df_swaps = pd.read_csv('swaps.csv',
                    index_col = 'Swaps',
                    usecols = ['Swaps','BID','ASK'],
                    converters={'Swaps':pd.to_datetime,'BID':swap_converter,'ASK':swap_converter})

today = dt.TARGET['Today']
settlement_date  = dt.TARGET['Settlement']

# Storing the data in a dictionary
market_data = dict()
market_data["reference_date"] = settlement_date
market_data["depo"] = df_depos
market_data["futures"] = df_futures
market_data["swaps"] = df_swaps
pickle.dump(market_data, open("market_data.p", "wb"))


# Bootstrap
discount_factors, zero_rates = bootstrap(settlement_date, df_depos, df_futures, df_swaps)

## Q1: Mark-to-Market the portfolio at the mid-rate curve


In [30]:
# =============================================================================
# Q1: Compute the forward swap rate (= swaption strike, since ATM)
# =============================================================================

# Swaption expiry: today + 10y1m
swaption_expiry = business_date_offset(
    today, year_offset=swaption_maturity_y, month_offset=swaption_maturity_m
)

# Underlying swap expiry: swaption expiry + 5y tenor
underlying_expiry = business_date_offset(
    today,
    year_offset=swaption_maturity_y + swaption_tenor_y,
    month_offset=swaption_maturity_m,
)

# Fixed leg schedule of the underlying forward-starting swap
swaption_underlying_fixed_leg_schedule = date_series(
    swaption_expiry, underlying_expiry, swaption_fixed_leg_freq
)

# Forward swap rate
fwd_swap_rate = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors,
    swaption_underlying_fixed_leg_schedule[0],
)
print(f"Forward swap rate: {fwd_swap_rate:.3%}")


Forward swap rate: 5.300%


In [40]:
# =============================================================================
# Q1: Portfolio MtM
# =============================================================================

strike = fwd_swap_rate #ATM swaption

swaption_price, swaption_delta = swaption_price_calculator(
    fwd_swap_rate,
    strike,
    settlement_date, #FIX: it was today
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors,
    swaption_type,
    compute_delta=True,
)

irs_expiry = business_date_offset(settlement_date, year_offset=irs_maturity)
irs_fixed_leg_payment_dates = date_series(settlement_date, irs_expiry, irs_fixed_leg_freq)[1:]

irs_rate = swap_par_rate(irs_fixed_leg_payment_dates, discount_factors)
irs_mtm = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors,irs_type
)

# Portfolio MtM = swaption value + IRS value
ptf_mtm = swaption_notional * swaption_price + irs_notional * irs_mtm
print(f"Swaption price (per unit notional): {swaption_price:.6f}")
print(f"IRS MtM (per unit notional):        {irs_mtm:.6f}")
print(f"Portfolio MtM:                       \u20ac{ptf_mtm:,.2f}")

Swaption price (per unit notional): 0.115926
IRS MtM (per unit notional):        0.000000
Portfolio MtM:                       €75,351,956.20


## Q2: Evaluate the portfolio DV01-parallel


In [41]:
# =============================================================================
# Q2: Portfolio DV01-parallel (numerical)
# =============================================================================

shock = 1e-4

# bootstrap returns a tuple (discount_factors, zero_rates)
# we unpack and discard zero_rates
discount_factors_up, _ = bootstrap(settlement_date, df_depos, df_futures, 
                                   df_swaps, shock)

fwd_swap_rate_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_up,
    swaption_underlying_fixed_leg_schedule[0],
)

# Swaption price under the shocked curve
swaption_price_up = swaption_price_calculator(
    fwd_swap_rate_up, # there was an error: no _up
    strike,
    settlement_date, #FIX: it was today
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_up,
    swaption_type,
)

irs_mtm_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_up, swap_type=irs_type
) #error: there was no _up

# Shocked portfolio MtM
ptf_mtm_up = swaption_notional * swaption_price_up + irs_notional * irs_mtm_up

# DV01-parallel
ptf_numeric_dv01 = ptf_mtm_up - ptf_mtm
print(f"Portfolio DV01-parallel: \u20ac{ptf_numeric_dv01:,.2f}")

Portfolio DV01-parallel: €-367,898.40


## Q3: Analytical portfolio DV01 approximation


In [ ]:
# =============================================================================
# Q3: Analytical portfolio DV01
# =============================================================================

irs_duration = irs_proxy_duration(
    settlement_date, irs_rate, irs_fixed_leg_payment_dates, discount_factors
)

ptf_proxy_dv01 = (
    swaption_notional * swaption_delta + irs_notional * irs_duration
) * 1e-4

print(f"Portfolio proxy DV01:    \u20ac{ptf_proxy_dv01:,.2f}")
print(f"Portfolio numeric DV01:  \u20ac{ptf_numeric_dv01:,.2f}")
print(f"Difference:             \u20ac{ptf_proxy_dv01 - ptf_numeric_dv01:,.2f}")

Portfolio proxy DV01:    €-294,900.42
Portfolio numeric DV01:  €-367,904.90
Difference:             €73,004.48


## Q4: Delta-hedge the portfolio with a 10y IRS


In [34]:
# =============================================================================
# Q4: Delta hedging with a 10y IRS
# =============================================================================

min_lot = 1_000_000  # IRS traded in multiples of €1M

# --- Numerical hedge ---
# N* = -DV01(portfolio) / DV01(10y IRS per €1M)
irs_dv01_per_1M = min_lot * irs_mtm_up 

delta_hedge_swap_notional = round(-ptf_numeric_dv01 / irs_dv01_per_1M) * min_lot

# Verify: hedged portfolio DV01 should be ~0
delta_hedge_dv01 = (
    swaption_notional * swaption_price_up + (delta_hedge_swap_notional + irs_notional) * irs_mtm_up
) - ptf_mtm # non c'era l'IRS iniziale
print(
    f"Numerical hedge: €{delta_hedge_swap_notional:,.0f} total IRS notional, residual DV01: €{delta_hedge_dv01:,.0f}"
)

# --- Analytical hedge (using the proxy DV01 from Q3) ---
# Same formula N* = -DV01(ptf) / DV01(10y IRS per €1M)
# but using proxy DV01 from Q3: DV01 per unit notional = irs_duration * 1bp
delta_hedge_swap_notional_proxy = round(
    -ptf_proxy_dv01 / (irs_duration * 1e-4) / min_lot
) * min_lot

print(
    f"Analytical hedge: €{delta_hedge_swap_notional_proxy:,.0f} total IRS notional"
)
print(
    "\nThe analytical approximation significantly overestimates the required hedge notional."
)

Numerical hedge: €-459,000,000 total IRS notional, residual DV01: €126
Analytical hedge: €-356,000,000 total IRS notional

The analytical approximation significantly overestimates the required hedge notional.


## Q5: Portfolio coarse-grained bucket DV01 (10y and 15y buckets)


In [35]:
# =============================================================================
# Q5: Coarse-Grained Bucket DV01 construction
# =============================================================================

# Effective maturities for weight calculation:
# - Depos/Swaps: index IS the maturity
# - Futures: index is quotation date, but they impact the curve at Expiry

depo_years = np.array([year_frac_act_x(settlement_date, d, 365) for d in df_depos.index])
future_years = np.array([year_frac_act_x(settlement_date, d, 365) for d in df_futures.index])
swap_years = np.array([year_frac_act_x(settlement_date, d, 365) for d in df_swaps.index])

years = np.concatenate([depo_years, future_years, swap_years])

# All pillar dates as bootstrap expects them (quotation dates for futures)
all_dates = df_depos.index.append(df_futures.index).append(df_swaps.index)

# 10y bucket: 1 up to 10y, linear to 0 at 15y
weights_10y = np.where(years <= 10, 1.0, np.where(years >= 15, 0.0, (15 - years) / 5))

# 15y bucket: 0 up to 10y, linear to 1 at 15y
weights_15y = np.where(years <= 10, 0.0, np.where(years >= 15, 1.0, (years - 10) / 5))

# Build shock Series, remove duplicates
shock_10y = pd.Series(weights_10y * 1e-4, index=all_dates)
shock_10y = shock_10y[~shock_10y.index.duplicated(keep='first')]

shock_15y = pd.Series(weights_15y * 1e-4, index=all_dates)
shock_15y = shock_15y[~shock_15y.index.duplicated(keep='first')]

# Re-bootstrap
discount_factors_10y_up, _ = bootstrap(settlement_date, df_depos, df_futures, df_swaps, shock_10y)
discount_factors_15y_up, _ = bootstrap(settlement_date, df_depos, df_futures, df_swaps, shock_15y)

In [42]:
# =============================================================================
# Q5: Portfolio Coarse-Grained 10y Bucket DV01
# =============================================================================

fwd_swap_rate_10y_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_10y_up,
    swaption_underlying_fixed_leg_schedule[0],
)

swaption_price_10y_up = swaption_price_calculator(
    fwd_swap_rate_10y_up,
    strike,
    settlement_date, #FIX: it was today
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_10y_up,
    swaption_type,
)

# IRS MtM under 10y bucket shock
irs_mtm_10y_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_10y_up, swap_type=irs_type
)

ptf_mtm_10y_up = (
    swaption_notional * swaption_price_10y_up + irs_notional * irs_mtm_10y_up
)

ptf_numeric_10y_dv01 = ptf_mtm_10y_up - ptf_mtm
print(f"Portfolio Coarse-Grained 10y Bucket DV01: \u20ac{ptf_numeric_10y_dv01:,.2f}")

Portfolio Coarse-Grained 10y Bucket DV01: €-924,212.55


In [43]:
# =============================================================================
# Q5: Portfolio Coarse-Grained 15y Bucket DV01
# =============================================================================

fwd_swap_rate_15y_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_15y_up,
    swaption_underlying_fixed_leg_schedule[0],
)

# Swaption price under 15y bucket shock
swaption_price_15y_up = swaption_price_calculator(
    fwd_swap_rate_15y_up,
    strike,
    settlement_date, #fix: it was today
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_15y_up,
    swaption_type,
)

# IRS MtM under 15y bucket shock
irs_mtm_15y_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_15y_up, swap_type = irs_type
)

ptf_mtm_15y_up = (
    swaption_notional * swaption_price_15y_up
    + irs_notional * irs_mtm_15y_up
)

ptf_numeric_15y_dv01 = ptf_mtm_15y_up - ptf_mtm
print(f"Portfolio Coarse-Grained 15y Bucket DV01: \u20ac{ptf_numeric_15y_dv01:,.2f}")

Portfolio Coarse-Grained 15y Bucket DV01: €556,823.66


## Q6: Delta-hedge with two liquid IRS (10y and 15y)


In [44]:
# =============================================================================
# Q6: Delta hedging with two IRS (10y and 15y)
# =============================================================================

# Choice: 10y and 15y IRS, matching the macro-buckets from Q5.
# Convention: RECEIVER, same as the original 550M IRS.
# N > 0 means additional receiver, N < 0 means effectively payer.

# --- IRS 1: 10y receiver ---
irs1_dv01_10y = swap_mtm(irs_rate, irs_fixed_leg_payment_dates, discount_factors_10y_up, swap_type=SwapType.RECEIVER)
irs1_dv01_15y = swap_mtm(irs_rate, irs_fixed_leg_payment_dates, discount_factors_15y_up, swap_type=SwapType.RECEIVER)

# --- IRS 2: 15y receiver ---
irs2_expiry = business_date_offset(settlement_date, year_offset=15)
irs2_fixed_leg_payment_dates = date_series(settlement_date, irs2_expiry, irs_fixed_leg_freq)[1:]
irs2_rate = swap_par_rate(irs2_fixed_leg_payment_dates, discount_factors)

irs2_dv01_10y = swap_mtm(irs2_rate, irs2_fixed_leg_payment_dates, discount_factors_10y_up, swap_type=SwapType.RECEIVER)
irs2_dv01_15y = swap_mtm(irs2_rate, irs2_fixed_leg_payment_dates, discount_factors_15y_up, swap_type=SwapType.RECEIVER)

# --- Solve 2x2 system ---
A = np.array([
    [irs1_dv01_10y, irs2_dv01_10y],
    [irs1_dv01_15y, irs2_dv01_15y],
])
b = -np.array([ptf_numeric_10y_dv01, ptf_numeric_15y_dv01])

N1, N2 = np.linalg.solve(A, b)

N1 = round(N1 / min_lot) * min_lot
N2 = round(N2 / min_lot) * min_lot

print(f"Hedge IRS 1 (10y): €{N1:,.0f} ({'receiver' if N1 > 0 else 'payer'})")
print(f"Hedge IRS 2 (15y): €{N2:,.0f} ({'receiver' if N2 > 0 else 'payer'})")

# --- Verify ---
residual_10y = ptf_numeric_10y_dv01 + N1 * irs1_dv01_10y + N2 * irs2_dv01_10y
residual_15y = ptf_numeric_15y_dv01 + N1 * irs1_dv01_15y + N2 * irs2_dv01_15y
print(f"\nResidual 10y bucket DV01: €{residual_10y:,.0f}")
print(f"Residual 15y bucket DV01: €{residual_15y:,.0f}")

Hedge IRS 1 (10y): €-1,155,000,000 (payer)
Hedge IRS 2 (15y): €518,000,000 (receiver)

Residual 10y bucket DV01: €357
Residual 15y bucket DV01: €-442


## Q7: Curve flattening scenario


In [45]:
# =============================================================================
# Q7: Curve flattening scenario
# =============================================================================

shock_flatten = shock_10y - shock_15y

discount_factors_flatten, _ = bootstrap(
    settlement_date, df_depos, df_futures, df_swaps, shock_flatten
)

# --- Reprice swaption ---
fwd_swap_rate_flatten = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_flatten,
    swaption_underlying_fixed_leg_schedule[0],
)

swaption_price_flatten = swaption_price_calculator(
    fwd_swap_rate_flatten, strike, today, swaption_expiry, underlying_expiry,
    sigma_black, swaption_fixed_leg_freq, discount_factors_flatten, swaption_type,
)

# --- Reprice IRS (all receiver, consistent with Q6) ---
irs_mtm_flatten = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_flatten, swap_type=SwapType.RECEIVER
)
irs2_mtm_flatten = swap_mtm(
    irs2_rate, irs2_fixed_leg_payment_dates, discount_factors_flatten, swap_type=SwapType.RECEIVER
)

# --- Strategy (a): swaption + (550M + hedge from Q4) 10y receiver ---
# delta_hedge_swap_notional from Q4 is also receiver convention
pnl_a = (
    swaption_notional * (swaption_price_flatten - swaption_price)
    + (irs_notional + delta_hedge_swap_notional) * irs_mtm_flatten
)
print(f"P&L strategy (a) - single 10y IRS hedge: €{pnl_a:,.2f}")

# --- Strategy (b): swaption + (550M + N1) 10y receiver + N2 15y receiver ---
pnl_b = (
    swaption_notional * (swaption_price_flatten - swaption_price)
    + (irs_notional + N1) * irs_mtm_flatten
    + N2 * irs2_mtm_flatten
)
print(f"P&L strategy (b) - two IRS hedge:        €{pnl_b:,.2f}")

print(f"\nStrategy (a) P&L: €{pnl_a:,.2f}")
print(f"Strategy (b) P&L: €{pnl_b:,.2f}")
print(
    "\nStrategy (b) is superior: by hedging both bucket DV01s independently, "
    "it protects against non-parallel curve moves like this flattening. "
    "Strategy (a) only neutralizes the total DV01 but remains exposed to "
    "curve reshaping risk."
)

P&L strategy (a) - single 10y IRS hedge: €-1,090,325.11
P&L strategy (b) - two IRS hedge:        €24,730.29

Strategy (a) P&L: €-1,090,325.11
Strategy (b) P&L: €24,730.29

Strategy (b) is superior: by hedging both bucket DV01s independently, it protects against non-parallel curve moves like this flattening. Strategy (a) only neutralizes the total DV01 but remains exposed to curve reshaping risk.
